# Exploraci?n y Pipeline de Entrenamiento (KDD Cup 2010 Algebra I)

Este notebook documenta el flujo real usado por el proyecto para:

1. Cargar datos (KDD + logs del juego).
2. Preprocesar y crear features (`FeatureEngineer`).
3. Entrenar y comparar modelos (`ModelTrainer`).
4. Guardar artefactos para la API y el panel admin.

> Recomendado: ejecutar primero `pip install -r requirements.txt`.

**Comandos equivalentes (CLI):**
- Entrenar: `python main.py train --no-plots`
- Servir API: `python main.py serve`


## 0) Setup

Este notebook asume que est? dentro de `notebooks/` y que el proyecto est? en el directorio padre.


In [ ]:
import sys
from pathlib import Path

ROOT = Path('..').resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print('Root:', ROOT)


## 1) Cargar dataset (KDD + logs)

El loader prioriza archivos `.txt` en `data/external/` (train/test/master) y adem?s incorpora `data/raw/gameplay_logs.csv` si existe.


In [ ]:
import pandas as pd

from src.utils.config import ProjectConfig
from src.data_processing.data_loader import DataLoader

config = ProjectConfig()
loader = DataLoader(config)

raw_df = loader.load_dataset()
print('Filas:', len(raw_df))
print('Columnas:', len(raw_df.columns))
raw_df.head(3)


In [ ]:
# Columnas esperadas por el pipeline del tutor
expected = [
    'student_id',
    'step_name',
    'incorrects',
    'hints',
    'correct_first_attempt',
    'step_duration_sec',
]

missing = [c for c in expected if c not in raw_df.columns]
print('Faltantes:', missing)

raw_df[expected].describe(include='all')


In [ ]:
# Distribuci?n de fuentes (train/test/master/user_gameplay)
if 'source_split' in raw_df.columns:
    raw_df['source_split'].value_counts(dropna=False)


## 2) Feature engineering (sin fuga de informaci?n)

`difficulty_level` se deriva de `difficulty_score`, pero el modelo **NO usa** `difficulty_score` como feature.
Eso evita la fuga m?s directa (m?trica artificialmente perfecta).


In [ ]:
from src.data_processing.feature_engineering import FeatureEngineer

engineer = FeatureEngineer()
dataset = engineer.transform(raw_df)

print('Filas (transform):', len(dataset))
print('Incluye target difficulty_level:', 'difficulty_level' in dataset.columns)
print('Incluye difficulty_score (solo para construir target):', 'difficulty_score' in dataset.columns)

x, y = engineer.split_features_target(dataset)
print('Features:', x.shape)
print('Target:', y.shape)

# Verificaci?n: difficulty_score no debe estar en X
print('difficulty_score en X:', 'difficulty_score' in x.columns)

x.head(3)


In [ ]:
# Distribuci?n del target (0=baja, 1=media, 2=alta)
y.value_counts(normalize=True).sort_index()


In [ ]:
# Algunas distribuciones r?pidas
cols = ['incorrects', 'hints', 'step_duration_sec', 'correct_first_attempt']
dataset[cols].describe()


## 3) Entrenamiento y validaci?n

El entrenamiento usa `ModelTrainer` y guarda:
- `models/model.pkl`
- `reports/metrics/model_leaderboard.csv`
- `reports/metrics/training_summary.json`

Validaci?n (orden de prioridad):
1) Si existe `source_split=train/test` y el test tiene variaci?n, respeta ese corte.
2) Si es posible, hace holdout por estudiante (`group_student_holdout_split`).
3) Si hay `event_order`, hace split temporal.
4) Caso final: split aleatorio.


In [ ]:
from src.models.model_trainer import ModelTrainer

trainer = ModelTrainer(config)

split_series = dataset['source_split'] if 'source_split' in dataset.columns else None
time_series = dataset['event_order'] if 'event_order' in dataset.columns else None
groups = dataset['student_id'] if 'student_id' in dataset.columns else None

# Para un notebook m?s r?pido, puedes reducir filas (por ejemplo: dataset = dataset.head(50000))
training_output = trainer.train(x, y, split_series=split_series, time_series=time_series, groups=groups)

leaderboard = training_output['leaderboard']
print('Validaci?n usada:', training_output['validation_type'])
leaderboard


## 4) M?tricas de aprendizaje desde logs (panel admin)

Estas m?tricas no miden el clasificador, sino la **mejora observable** del alumno (tiempo, pistas, accuracy, oscilaci?n de nivel, etc.).


In [ ]:
from src.app.learning_metrics import compute_learning_metrics_from_logs

metrics = compute_learning_metrics_from_logs(config.gameplay_log_path)
metrics['overall']


In [ ]:
# Top usuarios (m?ximo 50)
pd.DataFrame(metrics.get('per_user', []))


---

Actualizado: 2026-03-21
